# 04 — Completion and Open Tasks

Completion rates and elapsed times are separated from current open-task age.


In [ ]:
%run pathutils.ipynb
%run database.ipynb
%run export.ipynb
%run reporting-utils.ipynb

import sys
from datetime import date
from pathlib import Path
import matplotlib.pyplot as plt
import pandas as pd

sys.path.insert(0, str(Path(get_project_root_folder()) / "reports"))

In [ ]:
# Inclusive reporting period, based on task start date.
START_DATE = "2024-01-01"
END_DATE = "9999-12-31"
REPORT_DATE = date.today()

query = construct_query("task-history.sql", {"START-DATE": START_DATE, "END-DATE": END_DATE})
history = normalise_history(query_data(query))
period_label = f"{START_DATE} to {END_DATE} ({len(history):,} tasks)"
print(f"Reporting period: {period_label}")


## Completion rates

In [ ]:
completion_by_category = grouped_summary(history, "Category")
completion_by_type = grouped_summary(history, ["Category", "Task Type"])
completion_by_month = grouped_summary(history, "Start Month")
completion_by_category

## Elapsed time to completion

In [ ]:
completed = history.dropna(subset=["Completion Date"]).copy()
elapsed_summary = pd.DataFrame([{
    "Completed Tasks": len(completed), "Completed Same Day": int(completed["Elapsed Days"].eq(0).sum()),
    "Completed After Start Date": int(completed["Elapsed Days"].gt(0).sum()),
    "Median Elapsed Days": completed["Elapsed Days"].median(),
}])
elapsed_distribution = completed.groupby("Elapsed Days").size().rename("Task Count").reset_index()
elapsed_summary

In [ ]:
elapsed_distribution.plot.bar(x="Elapsed Days", y="Task Count", legend=False, title=f"Completion-time distribution — {period_label}")
plt.ylabel("Completed tasks"); plt.tight_layout(); plt.show()

## Current open tasks

In [ ]:
open_tasks = open_task_analysis(history, REPORT_DATE)
open_by_age = open_tasks.groupby("Age Band", observed=False).size().rename("Task Count").reset_index()
open_by_category = grouped_summary(open_tasks, "Category") if len(open_tasks) else pd.DataFrame()
open_by_type = grouped_summary(open_tasks, ["Category", "Task Type"]) if len(open_tasks) else pd.DataFrame()
open_by_age

In [ ]:
EXPORT_NAME = "04-completion-and-open-tasks.xlsx"
EXPORT_DATA = {"Category Completion": completion_by_category, "Type Completion": completion_by_type, "Monthly Completion": completion_by_month, "Elapsed Summary": elapsed_summary, "Elapsed Distribution": elapsed_distribution, "Open Tasks": open_tasks, "Open by Age": open_by_age, "Open by Category": open_by_category, "Open by Type": open_by_type}
export_to_spreadsheet(
    get_export_folder_path(), EXPORT_NAME, EXPORT_DATA
)
print(f"Exported {EXPORT_NAME} to {get_export_folder_path()}")
